# Playlist Matcher Test Notebook

This notebook creates a mock music library with the first 10 songs from Favourites.m3u8 and tests the playlist_matcher.py script.

## Setup

First, we'll install the required dependencies and import necessary modules.

In [2]:
# Install mutagen if not already installed
!pip install mutagen

In [3]:
import os
import shutil
from pathlib import Path
from mutagen.flac import FLAC
import re
import struct
from importlib import reload

import playlist_matcher as pm

## Helper Functions

Functions to sanitize filenames and handle special characters.

In [3]:
def sanitize_filename(name):
    """Sanitize a string for use in filenames by replacing problematic characters"""
    # Replace forward slash with unicode division slash or underscore
    # This preserves the visual appearance while being filesystem-safe
    name = name.replace('/', '∕')  # Unicode division slash U+2215
    # Could also use: name = name.replace('/', '_')
    
    # Replace other problematic characters
    replacements = {
        '\\': '∖',  # Backslash
        ':': '∶',   # Colon  
        '*': '∗',   # Asterisk
        '?': '？',  # Question mark
        '"': '＂',  # Quote
        '<': '＜',  # Less than
        '>': '＞',  # Greater than
        '|': '｜',  # Pipe
    }
    
    for old, new in replacements.items():
        name = name.replace(old, new)
    
    return name

def sanitize_path_component(name):
    """Sanitize directory/album names"""
    return sanitize_filename(name)

## Parse First 10 Entries from Favourites.m3u8

We'll read the playlist and extract metadata for the first 10 songs.

In [5]:
def parse_playlist_entries(playlist_path, num_entries=10):
    """Parse first N entries from playlist"""
    entries = []
    
    with open(playlist_path, 'r', encoding='utf-8-sig') as f:
        lines = [line.rstrip('\n\r') for line in f.readlines()]
    
    i = 0
    count = 0
    while i < len(lines) and count < num_entries:
        line = lines[i]
        
        if line.startswith('#EXTINF:'):
            if i + 1 < len(lines):
                extinf_line = line
                path_line = lines[i + 1]
                
                # Parse EXTINF: #EXTINF:duration,Artist - Title
                extinf_match = re.match(r'#EXTINF:(\d+),(.+)', extinf_line)
                if extinf_match:
                    duration = extinf_match.group(1)
                    artist_title = extinf_match.group(2)
                    
                    # Split artist and title
                    if ' - ' in artist_title:
                        artist, title = artist_title.split(' - ', 1)
                    else:
                        artist = ""
                        title = artist_title
                    
                    # Parse path: ..\Artist(s)\Album\CD# - Track# - Artist(s) - Title - Album.ext
                    clean_path = path_line.replace('..\\', '').replace('../', '').replace('\\', '/')
                    path_parts = clean_path.split('/')
                    
                    playlist_artist = path_parts[0] if len(path_parts) > 0 else artist
                    album = path_parts[1] if len(path_parts) > 1 else "Unknown Album"
                    filename = path_parts[2] if len(path_parts) > 2 else "unknown.flac"
                    
                    # Parse filename for additional metadata
                    # Format: CD# - Track# - Artist(s) - Title - Album.ext
                    # Use ' - ' (space-dash-space) as delimiter to allow '-' in names
                    filename_match = re.match(r'^(\d+) - (\d+) - (.+?) - (.+?) - (.+?)\.(\w+)$', filename)
                    
                    if filename_match:
                        disc = filename_match.group(1)
                        track = filename_match.group(2)
                        file_artist = filename_match.group(3)
                        file_title = filename_match.group(4)
                        file_album = filename_match.group(5)
                        ext = filename_match.group(6)
                    else:
                        disc = "1"
                        track = str(count + 1)
                        file_artist = artist
                        file_title = title
                        file_album = album
                        ext = "flac"
                    
                    entries.append({
                        'artist': artist.strip(),
                        'title': title.strip(),
                        'album': album.strip(),
                        'playlist_artist': playlist_artist.strip(),
                        'file_artist': file_artist.strip(),
                        'file_title': file_title.strip(),
                        'file_album': file_album.strip(),
                        'disc': disc,
                        'track': track,
                        'ext': ext,
                        'duration': duration,
                        'original_path': path_line
                    })
                    count += 1
                
                i += 2
            else:
                i += 1
        else:
            i += 1
    
    return entries

# Parse first 10 entries
entries = parse_playlist_entries('../Playlists/Favourites.m3u8', 10)

print(f"Parsed {len(entries)} entries:\n")
for i, entry in enumerate(entries, 1):
    print(f"{i}. {entry['artist']} - {entry['title']}")
    print(f"   Album: {entry['album']}")
    print(f"   Track: {entry['disc']}-{entry['track']}")
    print(f"   File title: {entry['file_title']}")
    print()

Parsed 10 entries:

1. The Offspring - (Can't Get My) Head Around You
   Album: Splinter
   Track: 1-164
   File title: (Can't Get My) Head Around You

2. Santana - (Da Le) Yaleo
   Album: Supernatural (Remastered)
   Track: 1-208
   File title: (Da Le) Yaleo

3. Jamiroquai - (Don't) Give Hate a Chance
   Album: Dynamite
   Track: 1-247
   File title: (Don't) Give Hate a Chance

4. The Rolling Stones - (I Can't Get No) Satisfaction - Mono
   Album: Out Of Our Heads
   Track: 1-458
   File title: (I Can't Get No) Satisfaction

5. JAY-Z, Beyoncé - 03' Bonnie & Clyde
   Album: The Blueprint 2 The Gift & The Curse
   Track: 1-1
   File title: 03' Bonnie & Clyde

6. Die Ärzte - 1/2 Lovesong
   Album: 13
   Track: 1-2
   File title: 12 Lovesong

7. Ciara, Missy Elliott - 1, 2 Step (feat. Missy Elliott)
   Album: Goodies
   Track: 1-3
   File title: 1, 2 Step (feat. Missy Elliott)

8. Gorillaz - 19-2000 - Soulchild Remix
   Album: Gorillaz
   Track: 1-4
   File title: 19-2000

9. A Tribe Call

## Create Mock Library Using Template

Now we'll copy the template FLAC file and modify its metadata for each song.
We sanitize filenames to handle special characters like '/' safely.

In [8]:
def create_mock_library(entries, template_path='example.flac', base_dir='test_music_library'):
    """Create mock music library by copying template and modifying metadata"""
    
    if not os.path.exists(template_path):
        print(f"Error: Template file {template_path} not found!")
        return []
    
    # Clean up existing directory
    if os.path.exists(base_dir):
        shutil.rmtree(base_dir)
    
    os.makedirs(base_dir)
    
    created_files = []
    
    for entry in entries:
        # Use artist as album artist for the directory structure
        album_artist = entry['artist']
        album = entry['album']
        
        # Sanitize directory names
        safe_artist = sanitize_path_component(album_artist)
        safe_album = sanitize_path_component(album)
        
        # Create directory structure
        album_dir = Path(base_dir) / safe_artist / safe_album
        album_dir.mkdir(parents=True, exist_ok=True)
        
        # Create filename in library format with sanitized components
        # Format: CD# - Track# - Title - Artist(s) - Album.ext
        safe_title = sanitize_filename(entry['title'])
        safe_artist_name = sanitize_filename(entry['artist'])
        safe_album_name = sanitize_filename(entry['album'])
        
        filename = f"{entry['disc']} - {entry['track']} - {safe_title} - {safe_artist_name} - {safe_album_name}.{entry['ext']}"
        file_path = album_dir / filename
        
        try:
            # Copy template file
            shutil.copy2(template_path, file_path)
            
            # Modify metadata - use ORIGINAL unsanitized values for metadata tags
            audio = FLAC(str(file_path))
            audio['title'] = entry['title']  # Original with /
            audio['artist'] = entry['artist']
            audio['album'] = entry['album']
            audio['albumartist'] = album_artist
            audio['tracknumber'] = entry['track']
            audio['discnumber'] = entry['disc']
            audio.save()
            
            # Verify
            audio = FLAC(str(file_path))
            print(f"✓ Created: {file_path.relative_to(base_dir)}")
            print(f"  Metadata: '{audio.get('title', [''])[0]}' by '{audio.get('artist', [''])[0]}'")
            
            created_files.append(str(file_path))
            
        except Exception as e:
            print(f"✗ Failed to create: {file_path.relative_to(base_dir) if file_path.exists() else filename}")
            print(f"  Error: {e}")
            import traceback
            traceback.print_exc()
    
    return created_files

# Create the mock library
print("Creating mock music library from template...\n")
created_files = create_mock_library(entries)
print(f"\nCreated {len(created_files)} files in test_music_library/")

Creating mock music library from template...

✓ Created: The Offspring\Splinter\1 - 164 - (Can't Get My) Head Around You - The Offspring - Splinter.flac
  Metadata: '(Can't Get My) Head Around You' by 'The Offspring'
✓ Created: Santana\Supernatural (Remastered)\1 - 208 - (Da Le) Yaleo - Santana - Supernatural (Remastered).flac
  Metadata: '(Da Le) Yaleo' by 'Santana'
✓ Created: Jamiroquai\Dynamite\1 - 247 - (Don't) Give Hate a Chance - Jamiroquai - Dynamite.flac
  Metadata: '(Don't) Give Hate a Chance' by 'Jamiroquai'
✓ Created: The Rolling Stones\Out Of Our Heads\1 - 458 - (I Can't Get No) Satisfaction - Mono - The Rolling Stones - Out Of Our Heads.flac
  Metadata: '(I Can't Get No) Satisfaction - Mono' by 'The Rolling Stones'
✓ Created: JAY-Z, Beyoncé\The Blueprint 2 The Gift & The Curse\1 - 1 - 03' Bonnie & Clyde - JAY-Z, Beyoncé - The Blueprint 2 The Gift & The Curse.flac
  Metadata: '03' Bonnie & Clyde' by 'JAY-Z, Beyoncé'
✓ Created: Die Ärzte\13\1 - 2 - 1∕2 Lovesong - Die Ärzte -

## Create Test Playlist

Create a small test playlist with just these 10 songs.

In [10]:
def create_test_playlist(entries, output_path='test_playlist.m3u8'):
    """Create a test playlist with the parsed entries"""
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('#EXTM3U\n')
        for entry in entries:
            f.write(f"#EXTINF:{entry['duration']},{entry['artist']} - {entry['title']}\n")
            f.write(f"{entry['original_path']}\n")
    
    print(f"Created test playlist: {output_path}")

create_test_playlist(entries)

Created test playlist: test_playlist.m3u8


## Test the Playlist Matcher Script

Now let's run the playlist matcher script on our test data.

In [11]:
# Run the playlist matcher
!python playlist_matcher.py \
    --playlist test_playlist.m3u8 \
    --music-dir test_music_library \
    --output test_output.m3u8 \
    --log test_unmatched.log \
    --format artist_album

2026-01-25 20:25:37,512 - INFO - Using playlist path format: artist_album
2026-01-25 20:25:37,512 - INFO - Format description: Artist(s)/Album/CD# - Track# - Artist(s) - Title - Album.ext
2026-01-25 20:25:37,512 - INFO - Step 1: Building library cache
2026-01-25 20:25:37,512 - INFO - Scanning music library: test_music_library
2026-01-25 20:25:37,513 - INFO - Processing album artist: 2Pac, Snoop Dogg
2026-01-25 20:25:38,094 - INFO - Processing album artist: A Tribe Called Quest
2026-01-25 20:25:38,095 - INFO - Processing album artist: Ciara, Missy Elliott
2026-01-25 20:25:38,096 - INFO - Processing album artist: Die Ärzte
2026-01-25 20:25:38,096 - INFO - Processing album artist: Gorillaz
2026-01-25 20:25:38,097 - INFO - Processing album artist: Jamiroquai
2026-01-25 20:25:38,097 - INFO - Processing album artist: JAY-Z, Beyoncé
2026-01-25 20:25:38,098 - INFO - Processing album artist: Santana
2026-01-25 20:25:38,099 - INFO - Processing album artist: The Offspring
2026-01-25 20:25:38,099 

## Verify Results

Let's check the output playlist and unmatched log.

In [12]:
print("=" * 80)
print("OUTPUT PLAYLIST (test_output.m3u8)")
print("=" * 80)
if os.path.exists('test_output.m3u8'):
    with open('test_output.m3u8', 'r', encoding='utf-8') as f:
        content = f.read()
        print(content)
        print(f"\nTotal matched songs: {content.count('#EXTINF')}")
else:
    print("File not found!")

print("\n" + "=" * 80)
print("UNMATCHED LOG (test_unmatched.log)")
print("=" * 80)
if os.path.exists('test_unmatched.log'):
    with open('test_unmatched.log', 'r', encoding='utf-8') as f:
        print(f.read())
else:
    print("File not found!")

OUTPUT PLAYLIST (test_output.m3u8)
#EXTM3U
#EXTINF:135,The Offspring - (Can't Get My) Head Around You
The Offspring\Splinter\1 - 164 - (Can't Get My) Head Around You - The Offspring - Splinter.flac
#EXTINF:352,Santana - (Da Le) Yaleo
Santana\Supernatural (Remastered)\1 - 208 - (Da Le) Yaleo - Santana - Supernatural (Remastered).flac
#EXTINF:300,Jamiroquai - (Don't) Give Hate a Chance
Jamiroquai\Dynamite\1 - 247 - (Don't) Give Hate a Chance - Jamiroquai - Dynamite.flac
#EXTINF:223,The Rolling Stones - (I Can't Get No) Satisfaction - Mono
The Rolling Stones\Out Of Our Heads\1 - 458 - (I Can't Get No) Satisfaction - Mono - The Rolling Stones - Out Of Our Heads.flac
#EXTINF:206,JAY-Z, Beyoncé - 03' Bonnie & Clyde
JAY-Z, Beyoncé\The Blueprint 2 The Gift & The Curse\1 - 1 - 03' Bonnie & Clyde - JAY-Z, Beyoncé - The Blueprint 2 The Gift & The Curse.flac
#EXTINF:235,Die Ärzte - 1/2 Lovesong
Die Ärzte\13\1 - 2 - 1∕2 Lovesong - Die Ärzte - 13.flac
#EXTINF:204,Ciara, Missy Elliott - 1, 2 Step (fe

## Verify File Structure

Let's verify the created library structure.

In [13]:
print("Test Music Library Structure:\n")
for root, dirs, files in os.walk('test_music_library'):
    level = root.replace('test_music_library', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in sorted(files):
        print(f"{subindent}{file}")

Test Music Library Structure:

test_music_library/
  2Pac, Snoop Dogg/
    All Eyez On Me/
      1 - 8 - 2 Of Amerikaz Most Wanted (ft. Snoop Doggy Dogg) - 2Pac, Snoop Dogg - All Eyez On Me.flac
  A Tribe Called Quest/
    From NYC/
      1 - 5 - 1nce again - A Tribe Called Quest - From NYC.flac
  Ciara, Missy Elliott/
    Goodies/
      1 - 3 - 1, 2 Step (feat. Missy Elliott) - Ciara, Missy Elliott - Goodies.flac
  Die Ärzte/
    13/
      1 - 2 - 1∕2 Lovesong - Die Ärzte - 13.flac
  Gorillaz/
    Gorillaz/
      1 - 4 - 19-2000 - Soulchild Remix - Gorillaz - Gorillaz.flac
  Jamiroquai/
    Dynamite/
      1 - 247 - (Don't) Give Hate a Chance - Jamiroquai - Dynamite.flac
  JAY-Z, Beyoncé/
    The Blueprint 2 The Gift & The Curse/
      1 - 1 - 03' Bonnie & Clyde - JAY-Z, Beyoncé - The Blueprint 2 The Gift & The Curse.flac
  Santana/
    Supernatural (Remastered)/
      1 - 208 - (Da Le) Yaleo - Santana - Supernatural (Remastered).flac
  The Offspring/
    Splinter/
      1 - 164 - (Ca

## Test Simple Text Playlist Format

Now let's test the script with a simple text playlist format (Artist - Title per line).

In [9]:
def create_text_playlist(entries, output_path='test_text_playlist.txt'):
    """Create a simple text playlist with Artist - Title format"""
    with open(output_path, 'w', encoding='utf-8') as f:
        for entry in entries:
            f.write(f"{entry['artist']} - {entry['title']}\n")
    
    print(f"Created text playlist: {output_path}")
    print(f"\nContents:")
    with open(output_path, 'r', encoding='utf-8') as f:
        content = f.read()
        print(content)
    return output_path

create_text_playlist(entries)

Created text playlist: test_text_playlist.txt

Contents:
The Offspring - (Can't Get My) Head Around You
Santana - (Da Le) Yaleo
Jamiroquai - (Don't) Give Hate a Chance
The Rolling Stones - (I Can't Get No) Satisfaction - Mono
JAY-Z, Beyoncé - 03' Bonnie & Clyde
Die Ärzte - 1/2 Lovesong
Ciara, Missy Elliott - 1, 2 Step (feat. Missy Elliott)
Gorillaz - 19-2000 - Soulchild Remix
A Tribe Called Quest - 1nce again
2Pac, Snoop Dogg - 2 Of Amerikaz Most Wanted (ft. Snoop Doggy Dogg)



'test_text_playlist.txt'

## Run Playlist Matcher on Text Format

Test the script with the simple text playlist.

In [10]:
# Run the playlist matcher on text format
!python3 playlist_matcher.py \
    --playlist test_text_playlist.txt \
    --music-dir test_music_library \
    --output test_text_output.m3u8 \
    --log test_text_unmatched.log \
    --format artist_album

2026-01-24 22:28:24,794 - INFO - Using playlist path format: artist_album
2026-01-24 22:28:24,794 - INFO - Format description: Artist(s)/Album/CD# - Track# - Artist(s) - Title - Album.ext
2026-01-24 22:28:24,794 - INFO - Step 1: Building library cache
2026-01-24 22:28:24,794 - INFO - Scanning music library: test_music_library
2026-01-24 22:28:24,794 - INFO - Processing album artist: 2Pac, Snoop Dogg
2026-01-24 22:28:24,806 - INFO - Processing album artist: A Tribe Called Quest
2026-01-24 22:28:24,807 - INFO - Processing album artist: Ciara, Missy Elliott
2026-01-24 22:28:24,809 - INFO - Processing album artist: Die Ärzte
2026-01-24 22:28:24,811 - INFO - Processing album artist: Gorillaz
2026-01-24 22:28:24,812 - INFO - Processing album artist: JAY-Z, Beyoncé
2026-01-24 22:28:24,814 - INFO - Processing album artist: Jamiroquai
2026-01-24 22:28:24,816 - INFO - Processing album artist: Santana
2026-01-24 22:28:24,818 - INFO - Processing album artist: The Offspring
2026-01-24 22:28:24,819 

## Verify Text Playlist Results

Check the output from the text playlist conversion.

In [ ]:
print("=" * 80)
print("TEXT PLAYLIST OUTPUT (test_text_output.m3u8)")
print("=" * 80)
if os.path.exists('test_text_output.m3u8'):
    with open('test_text_output.m3u8', 'r', encoding='utf-8') as f:
        content = f.read()
        print(content)
        print(f"\nTotal matched songs: {content.count('#EXTINF')}")
else:
    print("File not found!")

print("\n" + "=" * 80)
print("TEXT PLAYLIST UNMATCHED LOG (test_text_unmatched.log)")
print("=" * 80)
if os.path.exists('test_text_unmatched.log'):
    with open('test_text_unmatched.log', 'r', encoding='utf-8') as f:
        log_content = f.read()
        print(log_content)
        if 'Unmatched: 0' in log_content:
            print("\n✓ SUCCESS: All songs from text playlist were matched!")
else:
    print("File not found!")

## Summary

This notebook demonstrates:
1. Creating a mock music library with proper metadata
2. Testing M3U8 playlist format matching
3. Testing simple text playlist format (Artist - Title) matching
4. Verifying that both formats produce correct M3U8 output playlists

Both tests should show 100% match rate (10/10 songs matched).

In [4]:
reload(pm)
base = pm.PlaylistMatcher(
    playlist_path=Path(""),
    music_dir="E:\\Music",
    output_path=Path(""),
    log_path=Path("unmatched.log"),
    path_format="artist_album"
)

2026-01-26 09:11:48,543 - INFO - Using playlist path format: artist_album
2026-01-26 09:11:48,544 - INFO - Format description: Artist(s)/Album/CD# - Track# - Artist(s) - Title - Album.ext


In [5]:
base.build_library_cache()

2026-01-26 09:11:50,087 - INFO - Step 1: Building library cache
2026-01-26 09:11:50,088 - INFO - Scanning music library: E:\Music
2026-01-26 09:11:50,286 - INFO - Processing album artist: 19 Naughty III (US Release)
2026-01-26 09:11:50,302 - INFO - Processing album artist: 2 Brothers On The 4th Floor
2026-01-26 09:11:50,457 - INFO - Processing album artist: 2 Unlimited
2026-01-26 09:11:52,331 - INFO - Processing album artist: 2Pac
2026-01-26 09:11:55,126 - INFO - Cached 100 files...
2026-01-26 09:11:56,684 - INFO - Processing album artist: 3 Steps Ahead
2026-01-26 09:11:56,743 - INFO - Processing album artist: 4am Kru
2026-01-26 09:11:56,893 - INFO - Processing album artist: 50 Cent
2026-01-26 09:12:00,838 - INFO - Cached 200 files...
2026-01-26 09:12:02,984 - INFO - Processing album artist: 6 Pack (Explicit Version)
2026-01-26 09:12:03,031 - INFO - Processing album artist: 702
2026-01-26 09:12:03,090 - INFO - Processing album artist: 808 State
2026-01-26 09:12:03,227 - INFO - Processi

In [6]:
base.playlist_path = Path("E:\\rockbox\\Playlists\\Favourites.m3u8")
base.output_path = Path("E:\\rockbox\\library-playlist-parser\\Favourites.m3u8")
base.log_path = Path("E:\\rockbox\\library-playlist-parser\\unmatched.log")
base.path_prefix = ''

lines = base.read_old_playlist()
matched, unmatched = base.find_matches(lines)
base.write_new_playlist(matched)
base.write_log(matched, unmatched)

base.output_path = Path("E:\\Playlists\\Favourites.m3u8")
base.path_prefix = '..\\Music\\'
base.write_new_playlist(matched)
base.path_prefix = ''

2026-01-26 09:37:27,092 - INFO - Step 2: Reading playlist: E:\rockbox\Playlists\Favourites.m3u8
2026-01-26 09:37:27,408 - INFO - Read 2129 lines from playlist
2026-01-26 09:37:27,409 - INFO - Step 3: Finding matches for playlist entries
2026-01-26 09:37:27,409 - INFO - Detected playlist format: m3u8
2026-01-26 09:37:28,094 - INFO - Attempting partial match for artist 'Miami Sound Machine'
2026-01-26 09:37:28,095 - INFO -   Searching all tracks by artist 'Miami Sound Machine'
2026-01-26 09:37:28,095 - WARNING - ✗ No match: Miami Sound Machine - Conga!
2026-01-26 09:37:28,095 - WARNING -   Reason: No files found with title 'Conga!'; Found 1 file(s) with matching artist; No files found with album 'The Very Best Of Gloria Estefan (English Version)'; Artist exists but with different title
2026-01-26 09:37:28,971 - INFO - Attempting partial match for artist 'The Notorious B.I.G.'
2026-01-26 09:37:28,972 - INFO -   Searching all tracks by artist 'The Notorious B.I.G.'
2026-01-26 09:37:28,973 

In [7]:
base.playlist_path = Path("E:\\rockbox\\Playlists\\How Hard Can It Be.m3u8")
base.output_path = Path("E:\\rockbox\\library-playlist-parser\\How Hard Can It Be.m3u8")
base.log_path = Path("E:\\rockbox\\library-playlist-parser\\unmatched.log")
base.path_prefix = ''

lines = base.read_old_playlist()
matched, unmatched = base.find_matches(lines)
base.write_new_playlist(matched)
base.write_log(matched, unmatched)

base.output_path = Path("E:\\Playlists\\How Hard Can It Be.m3u8")
base.path_prefix = '..\\Music\\'
base.write_new_playlist(matched)
base.path_prefix = ''

2026-01-26 09:37:59,786 - INFO - Step 2: Reading playlist: E:\rockbox\Playlists\How Hard Can It Be.m3u8
2026-01-26 09:37:59,867 - INFO - Read 187 lines from playlist
2026-01-26 09:37:59,869 - INFO - Step 3: Finding matches for playlist entries
2026-01-26 09:37:59,869 - INFO - Detected playlist format: m3u8
2026-01-26 09:38:00,387 - INFO - Matched: 93, Unmatched: 0
2026-01-26 09:38:00,388 - INFO - Step 4: Writing new playlist: E:\rockbox\library-playlist-parser\How Hard Can It Be.m3u8
2026-01-26 09:38:00,389 - INFO - Wrote 93 entries to new playlist
2026-01-26 09:38:00,390 - INFO - Step 5: Writing unmatched log: E:\rockbox\library-playlist-parser\unmatched.log
2026-01-26 09:38:00,391 - INFO - 
2026-01-26 09:38:00,391 - INFO - SUMMARY
2026-01-26 09:38:00,391 - INFO - ================================================================================
2026-01-26 09:38:00,392 - INFO - Total songs: 93
2026-01-26 09:38:00,392 - INFO - Matched: 93
2026-01-26 09:38:00,392 - INFO - Unmatched: 0
202

In [8]:
base.playlist_path = Path("E:\\rockbox\\Playlists\\Wedding Party.m3u8")
base.output_path = Path("E:\\rockbox\\library-playlist-parser\\Wedding Party.m3u8")
base.log_path = Path("E:\\rockbox\\library-playlist-parser\\unmatched.log")
base.path_prefix = ''

lines = base.read_old_playlist()
matched, unmatched = base.find_matches(lines)
base.write_new_playlist(matched)
base.write_log(matched, unmatched)

base.output_path = Path("E:\\Playlists\\Wedding Party.m3u8")
base.path_prefix = '..\\Music\\'
base.write_new_playlist(matched)
base.path_prefix = ''

2026-01-26 09:38:44,241 - INFO - Step 2: Reading playlist: E:\rockbox\Playlists\Wedding Party.m3u8
2026-01-26 09:38:44,361 - INFO - Read 555 lines from playlist
2026-01-26 09:38:44,362 - INFO - Step 3: Finding matches for playlist entries
2026-01-26 09:38:44,362 - INFO - Detected playlist format: m3u8
2026-01-26 09:38:45,480 - INFO - Matched: 277, Unmatched: 0
2026-01-26 09:38:45,481 - INFO - Step 4: Writing new playlist: E:\rockbox\library-playlist-parser\Wedding Party.m3u8
2026-01-26 09:38:45,482 - INFO - Wrote 277 entries to new playlist
2026-01-26 09:38:45,483 - INFO - Step 5: Writing unmatched log: E:\rockbox\library-playlist-parser\unmatched.log
2026-01-26 09:38:45,483 - INFO - 
2026-01-26 09:38:45,484 - INFO - SUMMARY
2026-01-26 09:38:45,484 - INFO - ================================================================================
2026-01-26 09:38:45,484 - INFO - Total songs: 277
2026-01-26 09:38:45,484 - INFO - Matched: 277
2026-01-26 09:38:45,485 - INFO - Unmatched: 0
2026-01-2

In [9]:
base.playlist_path = Path("E:\\rockbox\\Playlists\\Japanese Sunset.m3u8")
base.output_path = Path("E:\\rockbox\\library-playlist-parser\\Japanese Sunset.m3u8")
base.log_path = Path("E:\\rockbox\\library-playlist-parser\\unmatched.log")
base.path_prefix = ''

lines = base.read_old_playlist()
matched, unmatched = base.find_matches(lines)
base.write_new_playlist(matched)
base.write_log(matched, unmatched)

base.output_path = Path("E:\\Playlists\\Japanese Sunset.m3u8")
base.path_prefix = '..\\Music\\'
base.write_new_playlist(matched)
base.path_prefix = ''

2026-01-26 09:38:52,042 - INFO - Step 2: Reading playlist: E:\rockbox\Playlists\Japanese Sunset.m3u8
2026-01-26 09:38:52,132 - INFO - Read 517 lines from playlist
2026-01-26 09:38:52,133 - INFO - Step 3: Finding matches for playlist entries
2026-01-26 09:38:52,133 - INFO - Detected playlist format: m3u8
2026-01-26 09:38:54,120 - INFO - Matched: 258, Unmatched: 0
2026-01-26 09:38:54,121 - INFO - Step 4: Writing new playlist: E:\rockbox\library-playlist-parser\Japanese Sunset.m3u8
2026-01-26 09:38:54,122 - INFO - Wrote 258 entries to new playlist
2026-01-26 09:38:54,122 - INFO - Step 5: Writing unmatched log: E:\rockbox\library-playlist-parser\unmatched.log
2026-01-26 09:38:54,123 - INFO - 
2026-01-26 09:38:54,123 - INFO - SUMMARY
2026-01-26 09:38:54,124 - INFO - ================================================================================
2026-01-26 09:38:54,124 - INFO - Total songs: 258
2026-01-26 09:38:54,124 - INFO - Matched: 258
2026-01-26 09:38:54,124 - INFO - Unmatched: 0
2026-

In [10]:
base.playlist_path = Path("E:\\rockbox\\Playlists\\So.txt")
base.output_path = Path("E:\\rockbox\\library-playlist-parser\\So.m3u8")
base.log_path = Path("E:\\rockbox\\library-playlist-parser\\unmatched.log")
base.path_prefix = ''

lines = base.read_old_playlist()
matched, unmatched = base.find_matches(lines)
base.write_new_playlist(matched)
base.write_log(matched, unmatched)

base.output_path = Path("E:\\Playlists\\So.m3u8")
base.path_prefix = '..\\Music\\'
base.write_new_playlist(matched)
base.path_prefix = ''

2026-01-26 09:38:59,553 - INFO - Step 2: Reading playlist: E:\rockbox\Playlists\So.txt
2026-01-26 09:38:59,590 - INFO - Read 225 lines from playlist
2026-01-26 09:38:59,590 - INFO - Step 3: Finding matches for playlist entries
2026-01-26 09:38:59,591 - INFO - Detected playlist format: text
2026-01-26 09:38:59,797 - WARNING - ✗ No match: Silk City,Dua Lipa,Mark Ronson,Diplo - Electricity
2026-01-26 09:38:59,798 - WARNING -   Reason: Found 2 file(s) with matching title; No files found with artist 'Silk City,Dua Lipa,Mark Ronson,Diplo'; Title exists but with different artist
2026-01-26 09:39:00,314 - WARNING - ✗ No match: Childish Gambino,Brittany Howard - Stay High - Childish Gambino Version
2026-01-26 09:39:00,315 - WARNING -   Reason: Found 1 file(s) with matching title; No files found with artist 'Childish Gambino,Brittany Howard'; Title exists but with different artist
2026-01-26 09:39:00,510 - INFO - Attempting partial match for artist 'Joesef'
2026-01-26 09:39:00,511 - INFO -   Sea

In [11]:
base.playlist_path = Path("E:\\rockbox\\Playlists\\Wee rock.txt")
base.output_path = Path("E:\\rockbox\\library-playlist-parser\\Wee rock.m3u8")
base.log_path = Path("E:\\rockbox\\library-playlist-parser\\unmatched.log")
base.path_prefix = ''

lines = base.read_old_playlist()
matched, unmatched = base.find_matches(lines)
base.write_new_playlist(matched)
base.write_log(matched, unmatched)

base.output_path = Path("E:\\Playlists\\Wee rock.m3u8")
base.path_prefix = '..\\Music\\'
base.write_new_playlist(matched)
base.path_prefix = ''

2026-01-26 09:39:01,820 - INFO - Step 2: Reading playlist: E:\rockbox\Playlists\Wee rock.txt
2026-01-26 09:39:01,970 - INFO - Read 48 lines from playlist
2026-01-26 09:39:01,970 - INFO - Step 3: Finding matches for playlist entries
2026-01-26 09:39:01,971 - INFO - Detected playlist format: text
2026-01-26 09:39:02,353 - INFO - Matched: 48, Unmatched: 0
2026-01-26 09:39:02,354 - INFO - Step 4: Writing new playlist: E:\rockbox\library-playlist-parser\Wee rock.m3u8
2026-01-26 09:39:02,355 - INFO - Wrote 48 entries to new playlist
2026-01-26 09:39:02,356 - INFO - Step 5: Writing unmatched log: E:\rockbox\library-playlist-parser\unmatched.log
2026-01-26 09:39:02,357 - INFO - 
2026-01-26 09:39:02,357 - INFO - SUMMARY
2026-01-26 09:39:02,357 - INFO - ================================================================================
2026-01-26 09:39:02,358 - INFO - Total songs: 48
2026-01-26 09:39:02,358 - INFO - Matched: 48
2026-01-26 09:39:02,358 - INFO - Unmatched: 0
2026-01-26 09:39:02,358 -

In [12]:
base.playlist_path = Path("E:\\rockbox\\Playlists\\Marvelous.txt")
base.output_path = Path("E:\\rockbox\\library-playlist-parser\\Marvelous.m3u8")
base.log_path = Path("E:\\rockbox\\library-playlist-parser\\unmatched.log")
base.path_prefix = ''

lines = base.read_old_playlist()
matched, unmatched = base.find_matches(lines)
base.write_new_playlist(matched)
base.write_log(matched, unmatched)

base.output_path = Path("E:\\Playlists\\Marvelous.m3u8")
base.path_prefix = '..\\Music\\'
base.write_new_playlist(matched)
base.path_prefix = ''

2026-01-26 09:39:03,648 - INFO - Step 2: Reading playlist: E:\rockbox\Playlists\Marvelous.txt
2026-01-26 09:39:03,658 - INFO - Read 217 lines from playlist
2026-01-26 09:39:03,659 - INFO - Step 3: Finding matches for playlist entries
2026-01-26 09:39:03,659 - INFO - Detected playlist format: text
2026-01-26 09:39:03,684 - WARNING - ✗ No match: Hazey Eyes,Panama - Emotion
2026-01-26 09:39:03,684 - WARNING -   Reason: Found 2 file(s) with matching title; No files found with artist 'Hazey Eyes,Panama'; Title exists but with different artist
2026-01-26 09:39:03,798 - WARNING - ✗ No match: Darlingside - Hold Your Head Up High - Recorded at Spotify Studios NYC
2026-01-26 09:39:03,799 - WARNING -   Reason: No files found with title 'Hold Your Head Up High - Recorded at Spotify Studios NYC'; No files found with artist 'Darlingside'
2026-01-26 09:39:03,930 - WARNING - ✗ No match: Sara Bareilles,Ingrid Michaelson - Winter Song
2026-01-26 09:39:03,930 - WARNING -   Reason: Found 2 file(s) with ma

In [13]:
base.playlist_path = Path("E:\\rockbox\\Playlists\\Christmas .txt")
base.output_path = Path("E:\\rockbox\\library-playlist-parser\\Christmas.m3u8")
base.log_path = Path("E:\\rockbox\\library-playlist-parser\\unmatched.log")
base.path_prefix = ''

lines = base.read_old_playlist()
matched, unmatched = base.find_matches(lines)
base.write_new_playlist(matched)
base.write_log(matched, unmatched)

base.output_path = Path("E:\\Playlists\\Christmas.m3u8")
base.path_prefix = '..\\Music\\'
base.write_new_playlist(matched)
base.path_prefix = ''

2026-01-26 09:39:05,991 - INFO - Step 2: Reading playlist: E:\rockbox\Playlists\Christmas .txt
2026-01-26 09:39:06,023 - INFO - Read 36 lines from playlist
2026-01-26 09:39:06,024 - INFO - Step 3: Finding matches for playlist entries
2026-01-26 09:39:06,024 - INFO - Detected playlist format: text
2026-01-26 09:39:06,125 - WARNING - ✗ No match: Ingrid Michaelson,Grace VanderWaal - Rockin' Around The Christmas Tree
2026-01-26 09:39:06,125 - WARNING -   Reason: Found 1 file(s) with matching title; No files found with artist 'Ingrid Michaelson,Grace VanderWaal'; Title exists but with different artist
2026-01-26 09:39:06,140 - WARNING - ✗ No match: Miley Cyrus,Mark Ronson,Sean Ono Lennon - Happy Xmas (War Is Over) (feat. Sean Ono Lennon)
2026-01-26 09:39:06,142 - WARNING -   Reason: Found 1 file(s) with matching title; No files found with artist 'Miley Cyrus,Mark Ronson,Sean Ono Lennon'; Title exists but with different artist
2026-01-26 09:39:06,232 - WARNING - ✗ No match: Bing Crosby,The A

In [14]:
base.playlist_path = Path("E:\\rockbox\\Playlists\\Nachtmusik.txt")
base.output_path = Path("E:\\rockbox\\library-playlist-parser\\Nachtmusik.m3u8")
base.log_path = Path("E:\\rockbox\\library-playlist-parser\\unmatched.log")
base.path_prefix = ''

lines = base.read_old_playlist()
matched, unmatched = base.find_matches(lines)
base.write_new_playlist(matched)
base.write_log(matched, unmatched)

base.output_path = Path("E:\\Playlists\\Nachtmusik.m3u8")
base.path_prefix = '..\\Music\\'
base.write_new_playlist(matched)
base.path_prefix = ''

2026-01-26 09:39:06,384 - INFO - Step 2: Reading playlist: E:\rockbox\Playlists\Nachtmusik.txt
2026-01-26 09:39:06,403 - INFO - Read 117 lines from playlist
2026-01-26 09:39:06,403 - INFO - Step 3: Finding matches for playlist entries
2026-01-26 09:39:06,404 - INFO - Detected playlist format: text
2026-01-26 09:39:06,743 - WARNING - ✗ No match: Broken Bells,Danger Mouse,James Mercer - Holding on for Life
2026-01-26 09:39:06,744 - WARNING -   Reason: Found 1 file(s) with matching title; No files found with artist 'Broken Bells,Danger Mouse,James Mercer'; Title exists but with different artist
2026-01-26 09:39:06,755 - WARNING - ✗ No match: Broken Bells,Danger Mouse,James Mercer - The High Road
2026-01-26 09:39:06,755 - WARNING -   Reason: Found 1 file(s) with matching title; No files found with artist 'Broken Bells,Danger Mouse,James Mercer'; Title exists but with different artist
2026-01-26 09:39:06,766 - WARNING - ✗ No match: Broken Bells,Danger Mouse,James Mercer - October
2026-01-26

In [15]:
base.playlist_path = Path("E:\\rockbox\\Playlists\\Sing Along .txt")
base.output_path = Path("E:\\rockbox\\library-playlist-parser\\Sing Along.m3u8")
base.log_path = Path("E:\\rockbox\\library-playlist-parser\\unmatched.log")
base.path_prefix = ''

lines = base.read_old_playlist()
matched, unmatched = base.find_matches(lines)
base.write_new_playlist(matched)
base.write_log(matched, unmatched)

base.output_path = Path("E:\\Playlists\\Sing Along.m3u8")
base.path_prefix = '..\\Music\\'
base.write_new_playlist(matched)
base.path_prefix = ''

2026-01-26 09:39:07,370 - INFO - Step 2: Reading playlist: E:\rockbox\Playlists\Sing Along .txt
2026-01-26 09:39:07,390 - INFO - Read 102 lines from playlist
2026-01-26 09:39:07,390 - INFO - Step 3: Finding matches for playlist entries
2026-01-26 09:39:07,391 - INFO - Detected playlist format: text
2026-01-26 09:39:07,464 - WARNING - ✗ No match: Whitney Houston,George Michael - If I Told You That (feat. George Michael) - Radio Edit
2026-01-26 09:39:07,465 - WARNING -   Reason: No files found with title 'If I Told You That (feat. George Michael) - Radio Edit'; No files found with artist 'Whitney Houston,George Michael'
2026-01-26 09:39:07,623 - WARNING - ✗ No match: Troy,Gabriella,Disney - Start of Something New
2026-01-26 09:39:07,623 - WARNING -   Reason: Found 1 file(s) with matching title; No files found with artist 'Troy,Gabriella,Disney'; Title exists but with different artist
2026-01-26 09:39:07,638 - WARNING - ✗ No match: Troy,Gabriella,Disney - Breaking Free
2026-01-26 09:39:07

In [16]:
base.playlist_path = Path("E:\\rockbox\\Playlists\\SAMURAI CHAMPLOO.txt")
base.output_path = Path("E:\\rockbox\\library-playlist-parser\\SAMURAI CHAMPLOO.m3u8")
base.log_path = Path("E:\\rockbox\\library-playlist-parser\\unmatched.log")
base.path_prefix = ''

hhcib_lines = base.read_old_playlist()
matched, unmatched = base.find_matches(hhcib_lines)
base.write_new_playlist(matched)
base.write_log(matched, unmatched)

base.output_path = Path("E:\\Playlists\\SAMURAI CHAMPLOO.m3u8")
base.path_prefix = '..\\Music\\'
base.write_new_playlist(matched)
base.path_prefix = ''

2026-01-26 09:39:11,189 - INFO - Step 2: Reading playlist: E:\rockbox\Playlists\SAMURAI CHAMPLOO.txt
2026-01-26 09:39:11,198 - INFO - Read 78 lines from playlist
2026-01-26 09:39:11,199 - INFO - Step 3: Finding matches for playlist entries
2026-01-26 09:39:11,200 - INFO - Detected playlist format: text
2026-01-26 09:39:11,227 - WARNING - ✗ No match: Nujabes,Shing02 - battlecry
2026-01-26 09:39:11,228 - WARNING -   Reason: Found 1 file(s) with matching title; No files found with artist 'Nujabes,Shing02'; Title exists but with different artist
2026-01-26 09:39:11,257 - INFO - Attempting partial match for artist 'MINMI'
2026-01-26 09:39:11,258 - INFO -   Searching all tracks by artist 'MINMI'
2026-01-26 09:39:11,258 - WARNING - ✗ No match: MINMI - 四季ノ唄
2026-01-26 09:39:11,259 - WARNING -   Reason: No files found with title '四季ノ唄'; Found 1 file(s) with matching artist; Artist exists but with different title
2026-01-26 09:39:11,289 - WARNING - ✗ No match: TSUTCHIE,kazami - YOU feat. kazami
